# TF-IDF + Logistic Regression — Department (Baseline)


In [91]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score

In [92]:
PROJECT_ROOT = Path.cwd().parents[1]
DATA_PATH = PROJECT_ROOT / "files"

df_dep = pd.read_csv(DATA_PATH / "department-v2.csv")

df_dep.shape, df_dep.columns

((10145, 2), Index(['text', 'label'], dtype='object'))

In [93]:
df_dep.head(5)

,text,label
0,Adjoint directeur communication,Marketing
1,Advisor Strategy and Projects,Project Management
2,Beratung & Projekte,Project Management
3,Beratung & Projektmanagement,Project Management
4,Beratung und Projektmanagement kommunale Partner,Project Management


In [94]:
df_dep.isna().sum().sort_values(ascending=False).head(20)

text     0
label    0
dtype: int64

In [95]:
df_dep["label"].value_counts()

label
Marketing                 4295
Sales                     3328
Information Technology    1305
Business Development       620
Project Management         201
Consulting                 167
Administrative              83
Other                       42
Purchasing                  40
Customer Support            33
Human Resources             31
Name: count, dtype: int64

Here i see that the Label Other Appears, in my Opinion it doesnt make sense to train the Modell on the Label Other, so we drop this.  After that we increased the accuracy on 2%.


In [96]:
df_dep = df_dep[df_dep["label"] != "Other"].reset_index(drop=True)

df_dep["label"].value_counts()

label
Marketing                 4295
Sales                     3328
Information Technology    1305
Business Development       620
Project Management         201
Consulting                 167
Administrative              83
Purchasing                  40
Customer Support            33
Human Resources             31
Name: count, dtype: int64

In [97]:
from sklearn.model_selection import train_test_split

X = df_dep["text"]
y = df_dep["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=0,
    stratify=y
)

X_train.shape, X_test.shape

((7072,), (3031,))

In [98]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

pipe_dep = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1,2),
        min_df=2,
        max_df=0.9,
        lowercase=True
    )),
    ("clf", LogisticRegression(
        max_iter=1000,
        n_jobs=-1,
        class_weight="balanced"
    ))
])

In [99]:
pipe_dep.fit(X_train, y_train)

Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_df=0.9, min_df=2, ngram_range=(1, 2))),
                ('clf',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    n_jobs=-1))])

In [100]:
from sklearn.metrics import classification_report

y_pred = pipe_dep.predict(X_test)
print(classification_report(y_test, y_pred))

                        precision    recall  f1-score   support

        Administrative       0.53      0.96      0.69        25
  Business Development       0.83      0.98      0.90       186
            Consulting       0.69      0.94      0.80        50
      Customer Support       0.69      0.90      0.78        10
       Human Resources       0.73      0.89      0.80         9
Information Technology       0.88      0.95      0.91       392
             Marketing       0.99      0.92      0.95      1289
    Project Management       0.60      0.93      0.73        60
            Purchasing       0.69      0.92      0.79        12
                 Sales       0.95      0.90      0.93       998

              accuracy                           0.92      3031
             macro avg       0.76      0.93      0.83      3031
          weighted avg       0.93      0.92      0.92      3031



In [101]:
pd.crosstab(
    y_test,
    y_pred,
    normalize="index",
    rownames=["truth"],
    colnames=["pred"]
).round(3)

pred,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Project Management,Purchasing,Sales
truth,,,,,,,,,,
Administrative,0.960,0.000,0.000,0.000,0.040,0.000,0.000,0.000,0.000,0.000
Business Development,0.000,0.978,0.000,0.000,0.000,0.011,0.000,0.005,0.000,0.005
Consulting,0.000,0.020,0.940,0.000,0.000,0.020,0.000,0.000,0.000,0.020
Customer Support,0.000,0.000,0.000,0.900,0.000,0.100,0.000,0.000,0.000,0.000
Human Resources,0.111,0.000,0.000,0.000,0.889,0.000,0.000,0.000,0.000,0.000
Information Technology,0.000,0.000,0.010,0.000,0.003,0.952,0.008,0.018,0.000,0.010
Marketing,0.009,0.012,0.004,0.002,0.001,0.011,0.918,0.014,0.002,0.028
Project Management,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.933,0.000,0.067
Purchasing,0.000,0.000,0.000,0.000,0.000,0.083,0.000,0.000,0.917,0.000


In [102]:
import json

with open(DATA_PATH / "linkedin-cvs-annotated.json", "r", encoding="utf-8") as f:
    cvs = json.load(f)

jobs_all = []
jobs_active = []

for person_id, cv in enumerate(cvs):
    for job in cv:
        row = {**job, "person_id": person_id}
        jobs_all.append(row)
        if job.get("status") == "ACTIVE":
            jobs_active.append(row)

df_all = pd.DataFrame(jobs_all)
df_active = pd.DataFrame(jobs_active)

df_active.shape, df_all.shape

((623, 9), (2638, 9))

In [103]:
def _norm_dep(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip()
    return x if x else np.nan

df_all["department_norm"] = df_all["department"].apply(_norm_dep)

In [104]:
hist = df_all[df_all["department_norm"].notna() & (df_all["department_norm"] != "Other")].copy()


In [105]:
hist_mode = (
    hist.groupby("person_id")["department_norm"]
        .agg(lambda s: s.value_counts().idxmax())
        .to_dict()
)

df_active["department_norm"] = df_active["department"].apply(_norm_dep)

In [106]:
df_active["department_gt_resolved"] = df_active["department_norm"]
needs_fix = df_active["department_gt_resolved"].isna() | (df_active["department_gt_resolved"] == "Other")
df_active.loc[needs_fix, "department_gt_resolved"] = df_active.loc[needs_fix, "person_id"].map(hist_mode)

print("\nResolved GT availability on ACTIVE:")
print(df_active["department_gt_resolved"].value_counts(dropna=False).head(20))


Resolved GT availability on ACTIVE:
department_gt_resolved
NaN                       164
Information Technology     96
Consulting                 78
Sales                      69
Project Management         58
Business Development       38
Marketing                  37
Administrative             29
Human Resources            26
Purchasing                 18
Customer Support           10
Name: count, dtype: int64


In [107]:
df_active["text_dep"] = (
    df_active["position"].fillna("") + " " +
    df_active["organization"].fillna("")
).str.lower()
df_active["department_pred_model"] = pipe_dep.predict(df_active["text_dep"])

print("\nModel can predict these departments (training labels):")
print(list(pipe_dep.named_steps["clf"].classes_))


Model can predict these departments (training labels):
['Administrative', 'Business Development', 'Consulting', 'Customer Support', 'Human Resources', 'Information Technology', 'Marketing', 'Project Management', 'Purchasing', 'Sales']


In [108]:
eval_mask = df_active["department_gt_resolved"].notna() & (df_active["department_gt_resolved"] != "Other")

acc_active = accuracy_score(
    df_active.loc[eval_mask, "department_gt_resolved"],
    df_active.loc[eval_mask, "department_pred_model"]
)

print("\n=== Evaluation on ACTIVE jobs (model vs resolved GT; no 'Other' in eval) ===")
print(f"Evaluated ACTIVE jobs: {eval_mask.sum()} / {len(df_active)}")
print(f"Accuracy: {acc_active:.4f}")


=== Evaluation on ACTIVE jobs (model vs resolved GT; no 'Other' in eval) ===
Evaluated ACTIVE jobs: 459 / 623
Accuracy: 0.3442


In [109]:
cm = pd.crosstab(
    df_active.loc[eval_mask, "department_gt_resolved"],
    df_active.loc[eval_mask, "department_pred_model"],
    normalize="index",
    rownames=["department_gt_resolved"],
    colnames=["department_pred_model"]
).round(3)

cm

department_pred_model,Administrative,Business Development,Consulting,Customer Support,Human Resources,Information Technology,Marketing,Project Management,Purchasing,Sales
department_gt_resolved,,,,,,,,,,
Administrative,0.103,0.000,0.000,0.000,0.000,0.103,0.276,0.103,0.000,0.414
Business Development,0.000,0.184,0.026,0.000,0.000,0.132,0.079,0.289,0.000,0.289
Consulting,0.000,0.090,0.218,0.013,0.026,0.128,0.051,0.103,0.000,0.372
Customer Support,0.000,0.000,0.000,0.100,0.000,0.100,0.100,0.100,0.000,0.600
Human Resources,0.038,0.000,0.000,0.000,0.308,0.000,0.000,0.115,0.000,0.538
Information Technology,0.000,0.021,0.000,0.010,0.000,0.271,0.021,0.250,0.000,0.427
Marketing,0.000,0.027,0.000,0.000,0.000,0.108,0.351,0.135,0.000,0.378
Project Management,0.000,0.017,0.017,0.000,0.000,0.086,0.086,0.414,0.000,0.379
Purchasing,0.111,0.000,0.000,0.000,0.000,0.111,0.111,0.056,0.333,0.278


In [110]:
df_active.loc[eval_mask, "department_pred_model"].value_counts(normalize=True).round(3)

department_pred_model
Sales                     0.451
Project Management        0.187
Information Technology    0.131
Marketing                 0.094
Consulting                0.041
Business Development      0.039
Human Resources           0.022
Purchasing                0.013
Administrative            0.013
Customer Support          0.009
Name: proportion, dtype: float64

In [111]:
df_active.loc[eval_mask, "department_gt_resolved"].value_counts(normalize=True).round(3)

department_gt_resolved
Information Technology    0.209
Consulting                0.170
Sales                     0.150
Project Management        0.126
Business Development      0.083
Marketing                 0.081
Administrative            0.063
Human Resources           0.057
Purchasing                0.039
Customer Support          0.022
Name: proportion, dtype: float64